# 03. Limpieza y preprocesamiento

Verificar la limpieza ya utilizada y conservar su archivo derivado.

**Dataset principal:** `data/nuevo_dataset/cardio_train.csv`. Este cuaderno no ejecuta entrenamientos. Los resultados históricos se leen de archivos conservados; no se sustituyen por nuevas ejecuciones.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/nuevo_dataset/cardio_train.csv").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from src.experimento import (
    ORIGINAL, RESULTADOS, VARIABLES, NOMBRES, cargar_datos, reglas_calidad,
    cargar_metricas, comparar_ejecuciones, tabla_configuracion, verificar_configuracion,
    construir_cnn, tabla_modelo, dibujar_arquitectura, guardar_figura_nueva,
    importancia_guardada, resumen_dataset, verificar_predicciones, tensorflow_disponible,
)

In [2]:
original = cargar_datos(ORIGINAL)
limpio_esperado = original.loc[~reglas_calidad(original).any(axis=1)].reset_index(drop=True)
ruta = RESULTADOS / "cardio_limpio.csv"
if ruta.exists():
    limpio = cargar_datos(ruta)
    pd.testing.assert_frame_equal(limpio, limpio_esperado, check_dtype=False)
    print("Dataset limpio existente verificado; no se sobrescribe.")
else:
    limpio_esperado.to_csv(ruta, sep=";", index=False)
    limpio = limpio_esperado
assert len(original) == 70000 and len(limpio) == 68615
print(f"{len(original):,} → {len(limpio):,}; excluidos: {len(original)-len(limpio):,}")

Dataset limpio existente verificado; no se sobrescribe.
70,000 → 68,615; excluidos: 1,385


## Representación para modelado
El CSV limpio conserva las 13 columnas originales y la edad en días para mantener trazabilidad. La conversión se aplica a una copia en memoria. `id` se conserva para rastrear registros, pero se excluye de X. `cardio` se separa como y. El escalador no se ajusta aquí: debe ajustarse después de separar entrenamiento y evaluación.

In [3]:
modelo = limpio.copy()
modelo["age"] = modelo["age"] / 365.25
X = modelo[VARIABLES].copy()
y = modelo["cardio"].copy()
assert X.shape[1] == 11
assert "id" not in X and "cardio" not in X
print("Predictoras:", X.columns.tolist())
print("Objetivo:", y.name)
display(X.describe().T)

Predictoras: ['age', 'gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']
Objetivo: cardio


,count,mean,std,min,25%,50%,75%,max
age,68615.0,53.291818,6.757009,29.563313,48.344969,53.941136,58.384668,64.922656
gender,68615.0,1.348641,0.476544,1.000000,1.000000,1.000000,2.000000,2.000000
height,68615.0,164.413277,7.912669,120.000000,159.000000,165.000000,170.000000,207.000000
weight,68615.0,74.114930,14.295824,30.000000,65.000000,72.000000,82.000000,200.000000
ap_hi,68615.0,126.676397,16.690374,70.000000,120.000000,120.000000,140.000000,240.000000
ap_lo,68615.0,81.308358,9.451607,40.000000,80.000000,80.000000,90.000000,182.000000
cholesterol,68615.0,1.364760,0.678946,1.000000,1.000000,1.000000,2.000000,3.000000
gluc,68615.0,1.225825,0.571674,1.000000,1.000000,1.000000,1.000000,3.000000
smoke,68615.0,0.087969,0.283252,0.000000,0.000000,0.000000,0.000000,1.000000
alco,68615.0,0.053356,0.224743,0.000000,0.000000,0.000000,0.000000,1.000000
